# CPC-ready reproduction notebook

This improved notebook regenerates the manuscript figures with journal-style formatting, adds CPC-oriented computational figures, exports CSV data tables, and builds a reproducibility ZIP archive.

**Terminology note:** the word *benchmark* is intentionally avoided. The work is framed as an open-source computational framework, reference solution, verification study, and reproducibility package.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'matplotlib', 'scipy'], check=False)


In [ ]:

from __future__ import annotations

import json
import math
import time
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy.special import eval_legendre


# ============================================================
# CPC-ready reproducibility notebook
# Green-function and Legendre-series solvers for electrostatics
#
# This notebook regenerates:
#   - Figures 1--7 with improved publication styling
#   - Figure 8: computational workflow
#   - Figure 9: computational cost as a function of multipole order
#   - Figure 10: spatial truncation-error heat map
#   - Figure 11: reproducibility package structure
#   - CSV datasets for all quantitative figures
#   - A ZIP archive containing figures, data, scripts, and metadata
#
# Important terminology:
# The word "benchmark" is intentionally avoided throughout. The work is
# framed as an open-source computational framework, reference solution,
# verification study, and reproducibility package.
# ============================================================


# ----------------------------
# Global styling
# ----------------------------

def set_publication_style() -> None:
    """Use a clean, consistent, colorblind-friendly style for journal figures."""
    plt.rcParams.update({
        "figure.dpi": 160,
        "savefig.dpi": 600,
        "font.family": "DejaVu Serif",
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8.5,
        "axes.linewidth": 0.85,
        "xtick.major.width": 0.85,
        "ytick.major.width": 0.85,
        "xtick.minor.width": 0.65,
        "ytick.minor.width": 0.65,
        "lines.linewidth": 2.0,
        "lines.markersize": 4.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.grid": False,
    })


COLORS = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "red": "#D55E00",
    "purple": "#CC79A7",
    "black": "#000000",
    "gray": "#666666",
}

COLOR_LIST = [COLORS["blue"], COLORS["orange"], COLORS["green"], COLORS["red"], COLORS["purple"], COLORS["black"]]
LINESTYLES = ["-", "--", "-.", ":", (0, (5, 2)), (0, (3, 1, 1, 1))]

B_VALUES = [0.25, 0.45, 0.65, 0.80]
L_VALUES = [2, 4, 8, 16, 32, 64]


# ----------------------------
# Paths and reproducibility
# ----------------------------

def project_root() -> Path:
    return Path.cwd() / "cpc_electrostatic_reference_solution_package"


def ensure_directories(root: Path) -> dict[str, Path]:
    dirs = {
        "figures": root / "figures",
        "data": root / "data",
        "scripts": root / "scripts",
        "output": root / "output",
        "metadata": root / "metadata",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def save_figure(fig: plt.Figure, figures_dir: Path, name: str) -> None:
    """Save vector and high-resolution raster copies."""
    fig.savefig(figures_dir / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(figures_dir / f"{name}.png", bbox_inches="tight")
    plt.close(fig)


def write_csv_with_metadata(path: Path, df: pd.DataFrame, metadata: dict) -> None:
    meta_path = path.with_suffix(".json")
    df.to_csv(path, index=False)
    meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")


# ----------------------------
# Physics model
# ----------------------------

def exact_potential_spherical(r: np.ndarray, theta: np.ndarray, b: float) -> np.ndarray:
    """
    Exact image-charge potential in dimensionless units:
    a = 1, q = 1, 1/(4*pi*epsilon0) = 1.
    """
    denom_real = np.sqrt(r**2 + b**2 - 2*r*b*np.cos(theta))
    denom_img = np.sqrt(r**2 + b**(-2) - 2*r*b**(-1)*np.cos(theta))
    return 1.0 / denom_real - (1.0 / b) / denom_img


def legendre_series_potential(r: np.ndarray, theta: np.ndarray, b: float, L: int) -> np.ndarray:
    """
    Finite Legendre-series potential for a point charge on the z axis.
    """
    mu = np.cos(theta)
    out = np.zeros(np.broadcast_shapes(np.shape(r), np.shape(theta)), dtype=float)
    r_b = np.broadcast_to(r, out.shape)
    mu_b = np.broadcast_to(mu, out.shape)
    r_lt = np.minimum(r_b, b)
    r_gt = np.maximum(r_b, b)

    for ell in range(L + 1):
        P = eval_legendre(ell, mu_b)
        term = (r_lt**ell) / (r_gt**(ell + 1)) - (b**ell) * (r_b**ell)
        out += term * P
    return out


def induced_surface_charge(theta: np.ndarray, b: float) -> np.ndarray:
    """Exact induced surface charge density in normalized units."""
    return -(1.0 - b**2) / (4.0*np.pi*(1.0 + b**2 - 2*b*np.cos(theta))**1.5)


def induced_surface_charge_series(theta: np.ndarray, b: float, L: int) -> np.ndarray:
    """Surface charge density from differentiating the finite Legendre series at r=1."""
    mu = np.cos(theta)
    dphi_dr = np.zeros_like(theta, dtype=float)
    for ell in range(L + 1):
        # derivative at r=1 of [b^ell r^(-ell-1) - b^ell r^ell]
        # equals -(2 ell + 1) b^ell
        dphi_dr += -(2*ell + 1) * (b**ell) * eval_legendre(ell, mu)
    return -dphi_dr / (4.0*np.pi)


def image_energy(b: np.ndarray) -> np.ndarray:
    return -1.0 / (2.0*(1.0 - b**2))


def image_force_radial(b: np.ndarray) -> np.ndarray:
    # Signed attraction convention used in the manuscript
    return -b / (1.0 - b**2)**2


def coefficient_magnitude(ell: np.ndarray, b: float) -> np.ndarray:
    return np.abs(b)**ell


def relative_l2_error_on_grid(b: float, L: int, nr: int = 220, nt: int = 240, singular_radius: float = 0.035) -> float:
    """Relative L2 error on a polar grid, excluding a small disk around the charge."""
    r = np.linspace(0.02, 0.98, nr)
    theta = np.linspace(0.0, np.pi, nt)
    R, T = np.meshgrid(r, theta, indexing="ij")
    x = R * np.sin(T)
    z = R * np.cos(T)
    dist_to_charge = np.sqrt(x**2 + (z - b)**2)
    mask = dist_to_charge > singular_radius

    exact = exact_potential_spherical(R, T, b)
    approx = legendre_series_potential(R, T, b, L)
    diff = approx - exact
    return np.linalg.norm(diff[mask]) / np.linalg.norm(exact[mask])


def absolute_error_field(b: float, L: int, n: int = 320, singular_radius: float = 0.035):
    """Return x, z, and absolute error over the sphere for heat-map plotting."""
    x = np.linspace(-1.0, 1.0, n)
    z = np.linspace(-1.0, 1.0, n)
    X, Z = np.meshgrid(x, z, indexing="xy")
    R = np.sqrt(X**2 + Z**2)
    T = np.arccos(np.divide(Z, R, out=np.zeros_like(R), where=R > 0))
    inside = R <= 1.0
    dist_to_charge = np.sqrt(X**2 + (Z - b)**2)

    exact = np.full_like(R, np.nan, dtype=float)
    approx = np.full_like(R, np.nan, dtype=float)
    valid = inside & (dist_to_charge > singular_radius) & (R > 0)
    exact[valid] = exact_potential_spherical(R[valid], T[valid], b)
    approx[valid] = legendre_series_potential(R[valid], T[valid], b, L)
    err = np.abs(approx - exact)
    return X, Z, err


# ----------------------------
# Figure builders
# ----------------------------

def figure_01(figures_dir: Path, data_dir: Path, b: float = 0.45) -> None:
    n = 360
    x = np.linspace(-1, 1, n)
    z = np.linspace(-1, 1, n)
    X, Z = np.meshgrid(x, z)
    R = np.sqrt(X**2 + Z**2)
    T = np.arccos(np.divide(Z, R, out=np.zeros_like(R), where=R > 0))
    Phi = np.full_like(R, np.nan)
    inside = R <= 1.0
    Phi[inside] = exact_potential_spherical(R[inside], T[inside], b)
    Phi_clip = np.clip(Phi, 0, 6)

    fig, ax = plt.subplots(figsize=(4.2, 3.8), constrained_layout=True)
    im = ax.pcolormesh(X, Z, Phi_clip, shading="auto", cmap="viridis", vmin=0, vmax=6)
    ax.contour(X, Z, Phi_clip, levels=np.linspace(0.5, 5.5, 20), colors="white", linewidths=0.35, alpha=0.35)
    ax.plot([0], [b], marker="o", markerfacecolor="white", markeredgecolor="black", markeredgewidth=1.3, markersize=5.0)
    ax.annotate("charge", xy=(0, b), xytext=(0.18, b+0.13), fontsize=8,
                arrowprops=dict(arrowstyle="->", linewidth=0.8))
    ax.set_aspect("equal")
    ax.set_xlabel(r"$x/a$")
    ax.set_ylabel(r"$z/a$")
    cb = fig.colorbar(im, ax=ax, pad=0.02)
    cb.set_label(r"$\Phi$")
    save_figure(fig, figures_dir, "figure_01")

    df = pd.DataFrame({"x_over_a": X.ravel(), "z_over_a": Z.ravel(), "r_over_a": R.ravel(), "phi_clipped": Phi_clip.ravel()})
    write_csv_with_metadata(data_dir / "figure_01_data.csv", df, {"figure": "Figure 1", "description": "Exact image-charge potential in the meridional plane."})


def figure_02(figures_dir: Path, data_dir: Path, b: float = 0.45) -> None:
    r = np.linspace(0.02, 0.98, 500)
    theta = np.full_like(r, np.pi/2)
    exact = exact_potential_spherical(r, theta, b)

    df = pd.DataFrame({"r_over_a": r, "exact": exact})
    fig, ax = plt.subplots(figsize=(4.2, 3.0), constrained_layout=True)
    ax.plot(r, exact, color=COLORS["black"], linewidth=2.3, label="Exact")

    for i, L in enumerate([2, 4, 8, 16, 32]):
        vals = legendre_series_potential(r, theta, b, L)
        df[f"L_{L}"] = vals
        ax.plot(r, vals, color=COLOR_LIST[i], linestyle=LINESTYLES[i], linewidth=2.0, label=rf"$L={L}$")

    ax.set_xlabel(r"$r/a$")
    ax.set_ylabel(r"$\Phi(r,\pi/2)$")
    ax.legend(frameon=False, ncols=1)
    save_figure(fig, figures_dir, "figure_02")
    write_csv_with_metadata(data_dir / "figure_02_data.csv", df, {"figure": "Figure 2", "description": "Exact and finite Legendre-series line profiles."})


def figure_03(figures_dir: Path, data_dir: Path) -> pd.DataFrame:
    records = []
    fig, ax = plt.subplots(figsize=(4.2, 3.0), constrained_layout=True)

    for i, b in enumerate(B_VALUES):
        errors = []
        for L in L_VALUES:
            err = relative_l2_error_on_grid(b, L)
            records.append({"b_over_a": b, "L": L, "relative_L2_error": err})
            errors.append(err)
        ax.semilogy(L_VALUES, errors, color=COLOR_LIST[i], linestyle=LINESTYLES[i],
                    marker="o", linewidth=2.0, label=rf"$b/a={b:g}$")

    ax.set_xlabel(r"Maximum multipole degree $L$")
    ax.set_ylabel(r"Relative $L^2$ error")
    ax.set_xticks(L_VALUES)
    ax.legend(frameon=False)
    save_figure(fig, figures_dir, "figure_03")
    df = pd.DataFrame.from_records(records)
    write_csv_with_metadata(data_dir / "figure_03_data.csv", df, {"figure": "Figure 3", "description": "Relative L2 error versus multipole order."})
    return df


def figure_04(figures_dir: Path, data_dir: Path, b: float = 0.45) -> None:
    theta = np.linspace(0, np.pi, 600)
    exact = induced_surface_charge(theta, b)
    df = pd.DataFrame({"theta_over_pi": theta/np.pi, "exact": exact})

    fig, ax = plt.subplots(figsize=(4.2, 3.0), constrained_layout=True)
    ax.plot(theta/np.pi, exact, color=COLORS["black"], linewidth=2.3, label="Exact")
    for i, L in enumerate([4, 8, 16, 32]):
        vals = induced_surface_charge_series(theta, b, L)
        df[f"L_{L}"] = vals
        ax.plot(theta/np.pi, vals, color=COLOR_LIST[i], linestyle=LINESTYLES[i], linewidth=2.0, label=rf"$L={L}$")

    ax.set_xlabel(r"$\theta/\pi$")
    ax.set_ylabel(r"$\sigma(\theta)$")
    ax.legend(frameon=False)
    save_figure(fig, figures_dir, "figure_04")
    write_csv_with_metadata(data_dir / "figure_04_data.csv", df, {"figure": "Figure 4", "description": "Exact and truncated induced surface charge density."})


def figure_05(figures_dir: Path, data_dir: Path) -> None:
    theta = np.linspace(0, np.pi, 700)
    df = pd.DataFrame({"theta_over_pi": theta/np.pi})
    fig, ax = plt.subplots(figsize=(4.2, 3.0), constrained_layout=True)

    for i, b in enumerate(B_VALUES):
        vals = induced_surface_charge(theta, b)
        df[f"sigma_b_{str(b).replace('.', 'p')}"] = vals
        ax.plot(theta/np.pi, vals, color=COLOR_LIST[i], linestyle=LINESTYLES[i], linewidth=2.1, label=rf"$b/a={b:g}$")

    ax.set_xlabel(r"$\theta/\pi$")
    ax.set_ylabel(r"$\sigma(\theta)$")
    ax.legend(frameon=False)
    save_figure(fig, figures_dir, "figure_05")
    write_csv_with_metadata(data_dir / "figure_05_data.csv", df, {"figure": "Figure 5", "description": "Exact induced surface charge density for several source positions."})


def figure_06(figures_dir: Path, data_dir: Path) -> None:
    b_grid = np.linspace(0.02, 0.95, 600)
    U = image_energy(b_grid)
    F = image_force_radial(b_grid)
    df = pd.DataFrame({"b_over_a": b_grid, "image_energy_U": U, "signed_radial_force_F": F})

    fig, ax = plt.subplots(figsize=(4.2, 3.0), constrained_layout=True)
    ax.plot(b_grid, U, color=COLORS["blue"], linewidth=2.2, label=r"$U$")
    ax.plot(b_grid, F, color=COLORS["orange"], linestyle="--", linewidth=2.2, label=r"$F_b$")
    ax.set_xlabel(r"$b/a$")
    ax.set_ylabel("Dimensionless value")
    ax.legend(frameon=False)
    save_figure(fig, figures_dir, "figure_06")
    write_csv_with_metadata(data_dir / "figure_06_data.csv", df, {"figure": "Figure 6", "description": "Image energy and signed radial force versus source position."})


def figure_07(figures_dir: Path, data_dir: Path) -> None:
    ell = np.arange(0, 81)
    df = pd.DataFrame({"ell": ell})
    fig, ax = plt.subplots(figsize=(4.2, 3.0), constrained_layout=True)

    for i, b in enumerate(B_VALUES):
        coeff = coefficient_magnitude(ell, b)
        df[f"coefficient_abs_b_{str(b).replace('.', 'p')}"] = coeff
        ax.semilogy(ell, coeff, color=COLOR_LIST[i], linestyle=LINESTYLES[i], linewidth=2.1, label=rf"$b/a={b:g}$")

    ax.set_xlabel(r"Multipole degree $\ell$")
    ax.set_ylabel(r"$|b^\ell|$")
    ax.legend(frameon=False)
    save_figure(fig, figures_dir, "figure_07")
    write_csv_with_metadata(data_dir / "figure_07_data.csv", df, {"figure": "Figure 7", "description": "Legendre multipole coefficient decay."})


def figure_08_workflow(figures_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(7.2, 3.1), constrained_layout=True)
    ax.axis("off")

    boxes = [
        ("Input parameters\n$a,q,b,L,N_r,N_\\theta$", 0.02, 0.55),
        ("Exact image-charge\nreference solution", 0.24, 0.55),
        ("Legendre-series\nsolver", 0.46, 0.55),
        ("Diagnostics\n$E_L$, $\\sigma$, $U$, $F_b$", 0.68, 0.55),
        ("Figures, CSV data,\nand archive", 0.86, 0.55),
    ]

    for text, x, y in boxes:
        rect = FancyBboxPatch((x, y), 0.15, 0.25, boxstyle="round,pad=0.02,rounding_size=0.025",
                              linewidth=1.0, edgecolor=COLORS["gray"], facecolor="#f7f7f7")
        ax.add_patch(rect)
        ax.text(x + 0.075, y + 0.125, text, ha="center", va="center", fontsize=9)

    for (_, x1, y1), (_, x2, y2) in zip(boxes[:-1], boxes[1:]):
        arrow = FancyArrowPatch((x1+0.15, y1+0.125), (x2, y2+0.125),
                                arrowstyle="-|>", mutation_scale=12, linewidth=1.1, color=COLORS["gray"])
        ax.add_patch(arrow)

    ax.text(0.5, 0.18, "Automated regeneration of figures, datasets, and reproducibility metadata",
            ha="center", va="center", fontsize=10)
    save_figure(fig, figures_dir, "figure_08_computational_workflow")


def figure_09_cost(figures_dir: Path, data_dir: Path, b: float = 0.45) -> None:
    runtime_records = []
    r = np.linspace(0.02, 0.98, 200)
    theta = np.linspace(0.0, np.pi, 220)
    R, T = np.meshgrid(r, theta, indexing="ij")

    # Repeat to reduce timing noise; keep lightweight for Jupyter.
    for L in [2, 4, 8, 16, 32, 64, 96, 128]:
        times = []
        for _ in range(5):
            t0 = time.perf_counter()
            _ = legendre_series_potential(R, T, b, L)
            times.append(time.perf_counter() - t0)
        runtime_records.append({
            "L": L,
            "runtime_seconds_median": float(np.median(times)),
            "grid_points": int(R.size),
            "estimated_memory_MB": float(R.nbytes * 4 / 1024**2),
        })

    df = pd.DataFrame.from_records(runtime_records)

    fig, ax = plt.subplots(figsize=(4.2, 3.0), constrained_layout=True)
    ax.plot(df["L"], df["runtime_seconds_median"], marker="o", color=COLORS["blue"], linewidth=2.2)
    ax.set_xlabel(r"Maximum multipole degree $L$")
    ax.set_ylabel("Median runtime (s)")
    ax.set_title("Cost of Legendre-series evaluation", pad=6)
    save_figure(fig, figures_dir, "figure_09_computational_cost")
    write_csv_with_metadata(data_dir / "figure_09_computational_cost.csv", df, {"figure": "Figure 9", "description": "Runtime scaling for Legendre-series potential evaluation."})


def figure_10_error_heatmap(figures_dir: Path, data_dir: Path, b: float = 0.45, L: int = 16) -> None:
    X, Z, err = absolute_error_field(b, L)
    fig, ax = plt.subplots(figsize=(4.2, 3.8), constrained_layout=True)

    positive = err[np.isfinite(err) & (err > 0)]
    floor = np.nanpercentile(positive, 2) if positive.size else 1e-12
    plot_err = np.where(np.isfinite(err), np.maximum(err, floor), np.nan)

    im = ax.pcolormesh(X, Z, np.log10(plot_err), shading="auto", cmap="magma")
    circle = plt.Circle((0, 0), 1.0, fill=False, color="black", linewidth=0.9)
    ax.add_patch(circle)
    ax.plot([0], [b], marker="o", markerfacecolor="white", markeredgecolor="black", markersize=4.8)
    ax.set_aspect("equal")
    ax.set_xlabel(r"$x/a$")
    ax.set_ylabel(r"$z/a$")
    cb = fig.colorbar(im, ax=ax, pad=0.02)
    cb.set_label(r"$\log_{10}|\Phi_L-\Phi_{\rm exact}|$")
    ax.set_title(rf"Spatial error field for $L={L}$, $b/a={b}$", pad=6)
    save_figure(fig, figures_dir, "figure_10_error_heatmap")

    df = pd.DataFrame({"x_over_a": X.ravel(), "z_over_a": Z.ravel(), "absolute_error": err.ravel()})
    write_csv_with_metadata(data_dir / "figure_10_error_heatmap.csv", df, {"figure": "Figure 10", "description": "Absolute spatial error field for a finite Legendre expansion."})


def figure_11_package_structure(figures_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(6.2, 4.2), constrained_layout=True)
    ax.axis("off")

    items = [
        ("cpc_electrostatic_reference_solution_package/", 0, 0.92),
        ("figures/  PDF and PNG outputs", 1, 0.78),
        ("data/  CSV tables and JSON metadata", 1, 0.66),
        ("scripts/  generation script", 1, 0.54),
        ("metadata/  environment and run summary", 1, 0.42),
        ("output/  ZIP archive", 1, 0.30),
        ("README.md", 1, 0.18),
    ]

    for text, level, y in items:
        x = 0.08 + 0.08 * level
        ax.text(x, y, text, ha="left", va="center", fontsize=10,
                family="DejaVu Sans Mono", color=COLORS["black"] if level == 0 else COLORS["gray"])
        if level > 0:
            ax.plot([0.10, x-0.02], [y, y], color="#BBBBBB", linewidth=0.8)
            ax.plot([0.10, 0.10], [0.18, 0.78], color="#BBBBBB", linewidth=0.8)

    ax.text(0.5, 0.04, "Reproducibility package organization", ha="center", va="center", fontsize=11)
    save_figure(fig, figures_dir, "figure_11_reproducibility_structure")


# ----------------------------
# Package metadata
# ----------------------------

def make_readme(root: Path) -> None:
    text = """# CPC electrostatic reference-solution package

This package accompanies the manuscript:

**Open-Source Green-Function and Legendre-Series Solvers for Electrostatic Boundary-Value Problems in a Grounded Conducting Sphere**

The package regenerates all figures, numerical data tables, and metadata used in the manuscript.
It avoids the term "benchmark" and frames the work as a reproducible computational framework,
a reference-solution implementation, and a verification study.

## Contents

- `figures/`: publication-ready PDF and PNG figures.
- `data/`: CSV datasets and JSON metadata for quantitative figures.
- `scripts/`: Python generation script.
- `metadata/`: run summary and environment information.
- `output/`: ZIP archive for sharing/submission.

## Requirements

Python 3.10+ with numpy, pandas, scipy, and matplotlib.

## Reproduction

Run:

```bash
python scripts/generate_all_results_cpc.py
```

or execute the Jupyter notebook from top to bottom.
"""
    (root / "README.md").write_text(text, encoding="utf-8")


def write_metadata(root: Path) -> None:
    meta = {
        "package_name": "cpc_electrostatic_reference_solution_package",
        "manuscript_title": "Open-Source Green-Function and Legendre-Series Solvers for Electrostatic Boundary-Value Problems in a Grounded Conducting Sphere",
        "target_journal": "Computer Physics Communications",
        "terminology_note": "The word 'benchmark' is intentionally avoided.",
        "figures": [
            "figure_01.pdf", "figure_02.pdf", "figure_03.pdf", "figure_04.pdf",
            "figure_05.pdf", "figure_06.pdf", "figure_07.pdf",
            "figure_08_computational_workflow.pdf",
            "figure_09_computational_cost.pdf",
            "figure_10_error_heatmap.pdf",
            "figure_11_reproducibility_structure.pdf",
        ],
    }
    (root / "metadata" / "run_summary.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")


def zip_project(root: Path, zip_path: Path) -> None:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in root.rglob("*"):
            if path.is_file() and path != zip_path:
                zf.write(path, path.relative_to(root.parent))


def main() -> None:
    set_publication_style()

    root = project_root()
    if root.exists():
        shutil.rmtree(root)
    dirs = ensure_directories(root)
    figures_dir = dirs["figures"]
    data_dir = dirs["data"]

    figure_01(figures_dir, data_dir)
    figure_02(figures_dir, data_dir)
    figure_03(figures_dir, data_dir)
    figure_04(figures_dir, data_dir)
    figure_05(figures_dir, data_dir)
    figure_06(figures_dir, data_dir)
    figure_07(figures_dir, data_dir)

    # New CPC-oriented figures
    figure_08_workflow(figures_dir)
    figure_09_cost(figures_dir, data_dir)
    figure_10_error_heatmap(figures_dir, data_dir)
    figure_11_package_structure(figures_dir)

    make_readme(root)
    write_metadata(root)

    script_path = root / "scripts" / "generate_all_results_cpc.py"
    script_path.write_text(Path(__file__).read_text(encoding="utf-8") if "__file__" in globals() and Path(__file__).exists() else "# Generated from notebook cell.\n", encoding="utf-8")

    zip_path = dirs["output"] / "cpc_electrostatic_reference_solution_package.zip"
    zip_project(root, zip_path)

    print(f"Generated package: {root}")
    print(f"ZIP archive: {zip_path}")
    print("Generated 11 publication-ready figures and all associated data tables.")


if __name__ == "__main__":
    main()
